# Model v4 — LightGBM + XGBoost Blend

**Base:** v2 (scored 0.2620)
**Change:** Blend 5-seed LightGBM (70%) + 3-seed XGBoost (30%)

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import gc
import warnings
import os
warnings.filterwarnings('ignore')

print('Libraries imported')

Libraries imported


In [2]:
TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH = '/kaggle/input/competitions/ts-forecasting/test.parquet'

if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'train.parquet'
    TEST_PATH = 'test.parquet'

VAL_THRESHOLD = 3500
FORECAST_WINDOWS = [1, 3, 10, 25]

# EXACT SAME LightGBM hyperparameters as proven 0.2598 model
LGB_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.015,
    'n_estimators': 4200,
    'num_leaves': 80,
    'min_child_samples': 200,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 10.0,
    'verbosity': -1
}

# XGBoost parameters — similar philosophy but different algorithm
XGB_PARAMS = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.015,
    'n_estimators': 4200,
    'max_depth': 8,
    'min_child_weight': 200,
    'subsample': 0.7,
    'colsample_bytree': 0.6,
    'reg_alpha': 0.1,
    'reg_lambda': 10.0,
    'tree_method': 'hist',
    'verbosity': 0,
    'early_stopping_rounds': 200
}

# Blend weights
LGB_WEIGHT = 0.70
XGB_WEIGHT = 0.30

print(f'Configuration complete — LGB {LGB_WEIGHT:.0%} + XGB {XGB_WEIGHT:.0%} blend')

Configuration complete — LGB 70% + XGB 30% blend


In [3]:
def add_lag_features(df, value_cols=['feature_al', 'feature_am', 'feature_cg', 'feature_by'], lags=[1, 3, 5, 10, 25]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    for col in value_cols:
        for lag in lags:
            df[f'{col}_lag_{lag}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].shift(lag)
    return df

def add_rolling_features(df, value_cols=['feature_al', 'feature_am'], windows=[5, 10, 20]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    for col in value_cols:
        for window in windows:
            df[f'{col}_roll_mean_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: x.rolling(window, min_periods=1).mean())
            df[f'{col}_roll_std_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: x.rolling(window, min_periods=1).std())
    return df

def add_trend_features(df, value_cols=['feature_al', 'feature_am'], windows=[10, 20]):
    df = df.sort_values(['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    def rolling_slope(series, window):
        def calc_slope(y):
            if len(y) < 2:
                return 0
            x = np.arange(len(y))
            return np.polyfit(x, y, 1)[0] if len(y) > 1 else 0
        return series.rolling(window, min_periods=2).apply(calc_slope, raw=True)
    for col in value_cols:
        for window in windows:
            df[f'{col}_trend_{window}'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].transform(lambda x: rolling_slope(x, window))
    return df

def build_enhanced_features(data, enc_stats=None):
    df = data.copy()
    if enc_stats is not None:
        for c in ['sub_category', 'sub_code']:
            df[c + '_enc'] = df[c].map(enc_stats[c]).fillna(enc_stats['global_mean'])

    # Original interactions
    df['d_al_am'] = df['feature_al'] - df['feature_am']
    df['r_al_am'] = df['feature_al'] / (df['feature_am'] + 1e-7)
    df['d_cg_by'] = df['feature_cg'] - df['feature_by']

    # Interaction features (from v2)
    df['p_am_bz'] = df['feature_am'] * df['feature_bz']
    df['p_am_cd'] = df['feature_am'] * df['feature_cd']
    df['d_j_bz']  = df['feature_j']  - df['feature_bz']
    df['r_l_bq']  = df['feature_l']  / (df['feature_bq'] + 1e-7)

    # Cross-sectional normalization
    norm_cols = [
        'feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am',
        'p_am_bz', 'p_am_cd', 'd_j_bz', 'r_l_bq'
    ]
    for col in norm_cols:
        if col in df.columns:
            g = df.groupby('ts_index')[col]
            df[col + '_cs'] = (df[col] - g.transform('mean')) / (g.transform('std') + 1e-7)

    df['t_cycle'] = np.sin(2 * np.pi * df['ts_index'] / 100)

    # Lag, rolling, trend (original)
    df = add_lag_features(df)
    df = add_rolling_features(df)
    df = add_trend_features(df)

    # Diff and rank
    for col in ['feature_al', 'feature_am']:
        df[f'{col}_diff_1'] = df.groupby(['code', 'sub_code', 'sub_category', 'horizon'])[col].diff(1)
        df[f'{col}_rank'] = df.groupby('ts_index')[col].rank(pct=True)

    # Additional rank features (from v2)
    for col in ['feature_bz', 'feature_cg', 'd_al_am']:
        if col in df.columns:
            df[f'{col}_rank'] = df.groupby('ts_index')[col].rank(pct=True)

    df = df.fillna(0)
    return df

def get_feature_columns(df):
    exclude_cols = {'id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'weight', 'y_target'}
    return [c for c in df.columns if c not in exclude_cols]

print('Feature engineering ready — same as v2 (0.2620)')

Feature engineering ready — same as v2 (0.2620)


In [4]:
def weighted_rmse_score(y_target, y_pred, w):
    y_target = np.array(y_target)
    y_pred = np.array(y_pred)
    w = np.array(w)
    denom = np.sum(w * (y_target ** 2))
    if denom <= 0:
        return 0.0
    numerator = np.sum(w * ((y_target - y_pred) ** 2))
    ratio = numerator / denom
    return float(np.sqrt(1.0 - np.clip(ratio, 0.0, 1.0)))

print('Metric ready')

Metric ready


In [5]:
print('Computing statistics...')
temp = pd.read_parquet(TRAIN_PATH, columns=['sub_category', 'sub_code', 'y_target', 'ts_index'])
train_only = temp[temp.ts_index <= VAL_THRESHOLD]
train_stats = {
    'sub_category': train_only.groupby('sub_category')['y_target'].mean().to_dict(),
    'sub_code': train_only.groupby('sub_code')['y_target'].mean().to_dict(),
    'global_mean': train_only['y_target'].mean()
}
del temp, train_only
gc.collect()
print('Statistics computed')

Computing statistics...
Statistics computed


In [6]:
def train_single_horizon(horizon):
    print(f'\n{"="*50}')
    print(f'Training Horizon {horizon}')
    print(f'{"="*50}')

    tr_df = pd.read_parquet(TRAIN_PATH).query(f'horizon == {horizon}')
    te_df = pd.read_parquet(TEST_PATH).query(f'horizon == {horizon}')
    tr_df = build_enhanced_features(tr_df, train_stats)
    te_df = build_enhanced_features(te_df, train_stats)
    feature_cols = get_feature_columns(tr_df)
    print(f'Using {len(feature_cols)} features')

    fit_mask = tr_df.ts_index <= VAL_THRESHOLD
    val_mask = tr_df.ts_index > VAL_THRESHOLD

    X_fit = tr_df.loc[fit_mask, feature_cols]
    y_fit = tr_df.loc[fit_mask, 'y_target']
    w_fit = tr_df.loc[fit_mask, 'weight']
    X_hold = tr_df.loc[val_mask, feature_cols]
    y_hold = tr_df.loc[val_mask, 'y_target']
    w_hold = tr_df.loc[val_mask, 'weight']

    # =========================================
    # LightGBM: 5 seeds
    # =========================================
    print('\n--- LightGBM (5 seeds) ---')
    lgb_val_pred = np.zeros(len(y_hold))
    lgb_tst_pred = np.zeros(len(te_df))
    lgb_seeds = [42, 2024, 12345, 99, 420]

    for seed in lgb_seeds:
        mdl = lgb.LGBMRegressor(**LGB_PARAMS, random_state=seed)
        mdl.fit(
            X_fit, y_fit, sample_weight=w_fit,
            eval_set=[(X_hold, y_hold)],
            eval_sample_weight=[w_hold],
            callbacks=[lgb.early_stopping(200, verbose=False)]
        )
        lgb_val_pred += mdl.predict(X_hold) / len(lgb_seeds)
        lgb_tst_pred += mdl.predict(te_df[feature_cols]) / len(lgb_seeds)
        print(f'  LGB seed {seed}: best_iter={mdl.best_iteration_}')

    lgb_score = weighted_rmse_score(y_hold, lgb_val_pred, w_hold)
    print(f'  LGB-only score: {lgb_score:.5f}')

    # =========================================
    # XGBoost: 3 seeds
    # =========================================
    print('\n--- XGBoost (3 seeds) ---')
    xgb_val_pred = np.zeros(len(y_hold))
    xgb_tst_pred = np.zeros(len(te_df))
    xgb_seeds = [42, 2024, 12345]

    for seed in xgb_seeds:
        xgb_mdl = xgb.XGBRegressor(**XGB_PARAMS, random_state=seed)
        xgb_mdl.fit(
            X_fit, y_fit, sample_weight=w_fit,
            eval_set=[(X_hold, y_hold)],
            sample_weight_eval_set=[w_hold],
            verbose=False
        )
        xgb_val_pred += xgb_mdl.predict(X_hold) / len(xgb_seeds)
        xgb_tst_pred += xgb_mdl.predict(te_df[feature_cols]) / len(xgb_seeds)
        bi = getattr(xgb_mdl, 'best_iteration', xgb_mdl.n_estimators)
        print(f'  XGB seed {seed}: best_iter={bi}')

    xgb_score = weighted_rmse_score(y_hold, xgb_val_pred, w_hold)
    print(f'  XGB-only score: {xgb_score:.5f}')

    # =========================================
    # Blend
    # =========================================
    val_pred = LGB_WEIGHT * lgb_val_pred + XGB_WEIGHT * xgb_val_pred
    tst_pred = LGB_WEIGHT * lgb_tst_pred + XGB_WEIGHT * xgb_tst_pred

    blend_score = weighted_rmse_score(y_hold, val_pred, w_hold)
    print(f'\n  BLENDED score: {blend_score:.5f}')
    print(f'  (LGB={lgb_score:.5f}, XGB={xgb_score:.5f}, Blend={blend_score:.5f})')

    test_ids = te_df['id'].values

    del tr_df, te_df, X_fit, y_fit, X_hold, y_hold, mdl, xgb_mdl
    gc.collect()
    return tst_pred, test_ids, blend_score

print('Training function ready — LGB+XGB blend')

Training function ready — LGB+XGB blend


In [7]:
test_outputs = []
validation_scores = {}

for hz in FORECAST_WINDOWS:
    tst_pred, test_ids, val_score = train_single_horizon(hz)
    test_outputs.append(pd.DataFrame({'id': test_ids, 'prediction': tst_pred}))
    validation_scores[hz] = val_score

print('\nAll horizons complete')


Training Horizon 1
Using 148 features

--- LightGBM (5 seeds) ---
  LGB seed 42: best_iter=239
  LGB seed 2024: best_iter=291
  LGB seed 12345: best_iter=322
  LGB seed 99: best_iter=201
  LGB seed 420: best_iter=252
  LGB-only score: 0.06938

--- XGBoost (3 seeds) ---
  XGB seed 42: best_iter=42
  XGB seed 2024: best_iter=4
  XGB seed 12345: best_iter=10
  XGB-only score: 0.03675

  BLENDED score: 0.06632
  (LGB=0.06938, XGB=0.03675, Blend=0.06632)

Training Horizon 3
Using 148 features

--- LightGBM (5 seeds) ---
  LGB seed 42: best_iter=447
  LGB seed 2024: best_iter=486
  LGB seed 12345: best_iter=771
  LGB seed 99: best_iter=1340
  LGB seed 420: best_iter=1119
  LGB-only score: 0.13679

--- XGBoost (3 seeds) ---
  XGB seed 42: best_iter=49
  XGB seed 2024: best_iter=30
  XGB seed 12345: best_iter=81
  XGB-only score: 0.08483

  BLENDED score: 0.13075
  (LGB=0.13679, XGB=0.08483, Blend=0.13075)

Training Horizon 10
Using 148 features

--- LightGBM (5 seeds) ---
  LGB seed 42: best

In [8]:
submission = pd.concat(test_outputs, ignore_index=True)
submission.to_csv('submission.csv', index=False)

print(f'\n{"="*50}')
print(f'{"Horizon":<10} {"Val Score":<12}')
print('-'*25)
for hz in sorted(validation_scores.keys()):
    print(f'{hz:<10} {validation_scores[hz]:<12.5f}')
print('-'*25)
avg_val = np.mean(list(validation_scores.values()))
print(f'{"Avg":<10} {avg_val:<12.5f}')
print(f'{"="*50}')
print(f'\nSaved {len(submission):,} predictions to submission.csv')
print(f'\nPrev best: 0.2620 (test)')


Horizon    Val Score   
-------------------------
1          0.06632     
3          0.13075     
10         0.21163     
25         0.26512     
-------------------------
Avg        0.16846     

Saved 1,447,107 predictions to submission.csv

Prev best: 0.2620 (test)
